# 04 — GitHub Community Usage Evaluation

The core methodological contribution: evaluating translation quality not by
back-translation against a reference (which assumes a ground truth exists)
but by comparing pipeline outputs to how scholarly communities actually name
these concepts in their practice.

This notebook analyses the GitHub alignment results and builds the evidence
for the paper's argument that:
  1. Pipeline outputs do not reliably reflect community usage for lower-resource
     language communities
  2. Some communities have adopted the English term as a loan word, which is a
     meaningful community choice that the pipeline cannot detect
  3. The distribution of GitHub signal across languages is itself evidence about
     which communities are represented in global DH discourse

**Run order:** After `evaluate_github_alignment.py`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from data_generation_scripts.utils import get_data_directory_path, read_csv_file

DATA_DIR = get_data_directory_path()
EVAL_DIR = os.path.join(DATA_DIR, 'metadata_files', 'evaluation')
VARIANTS = ['comparative', 'minimal', 'expert_persona', 'contextual', 'native_rationale']

SIGNAL_COLOURS = {
    'strong': '#2ca02c',
    'weak':   '#98df8a',
    'loan':   '#ff7f0e',
    'absent': '#d62728',
    'no_translation': '#aec7e8',
}

# Load evaluation outputs
align_path   = os.path.join(EVAL_DIR, 'github_alignment.csv')
summary_path = os.path.join(EVAL_DIR, 'github_alignment_summary.csv')
absent_path  = os.path.join(EVAL_DIR, 'absent_concept_candidates.csv')
loan_path    = os.path.join(EVAL_DIR, 'loan_word_cases.csv')

align_df   = read_csv_file(align_path)   if os.path.exists(align_path)   else None
summary_df = read_csv_file(summary_path) if os.path.exists(summary_path) else None
absent_df  = read_csv_file(absent_path)  if os.path.exists(absent_path)  else None
loan_df    = read_csv_file(loan_path)    if os.path.exists(loan_path)    else None

for name, df in [('Alignment', align_df), ('Summary', summary_df),
                  ('Absent', absent_df), ('Loan words', loan_df)]:
    print(f'{name}: {len(df) if df is not None else "NOT FOUND"} rows')

## 4.1 Overall Signal Distribution

How many languages fall into each alignment category?

In [ ]:
if align_df is not None:
    # Use comparative variant as representative
    signal_col = 'comparative_signal'
    if signal_col in align_df.columns:
        counts = align_df[signal_col].value_counts()
        total = len(align_df)
        print('Signal distribution (comparative variant):')
        for signal, count in counts.items():
            bar = '█' * int(count / total * 40)
            print(f'  {signal:15s} {bar} {count:3d} ({round(count/total*100,1)}%)')
        
        # Pie chart
        fig, ax = plt.subplots(figsize=(7, 7))
        colours = [SIGNAL_COLOURS.get(s, 'grey') for s in counts.index]
        wedges, texts, autotexts = ax.pie(
            counts.values, labels=counts.index, colors=colours,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8
        )
        for text in texts + autotexts:
            text.set_fontsize(11)
        ax.set_title('GitHub Alignment Signal Distribution\n(Comparative Variant)', pad=15)
        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_DIR, '04_signal_distribution.png'), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f'Column {signal_col} not found. Available:', list(align_df.columns[:10]))

## 4.2 Signal Distribution by Language Family

Does alignment with community usage vary systematically by language family?
Expected finding: higher alignment for Romance/Germanic (over-represented in
DH discourse) and lower for African, South/Southeast Asian language families.

In [ ]:
if summary_df is not None:
    # Focus on comparative variant
    comp_summary = summary_df[summary_df['variant'] == 'comparative']\
                   if 'variant' in summary_df.columns else summary_df
    
    if len(comp_summary) > 0:
        # Stacked bar: signal categories per family
        signal_cols = ['strong_match', 'weak_match', 'loan_word', 'absent']
        available = [c for c in signal_cols if c in comp_summary.columns]
        
        if available and 'language_family' in comp_summary.columns:
            plot_df = comp_summary.set_index('language_family')[available]
            
            fig, ax = plt.subplots(figsize=(10, 5))
            colours = [SIGNAL_COLOURS.get(c.replace('_match','').replace('_word',''), 'grey')
                       for c in available]
            plot_df.plot(kind='bar', stacked=True, ax=ax, color=colours, edgecolor='white')
            ax.set_title('GitHub Alignment by Language Family (Comparative Variant)')
            ax.set_xlabel('Language Family')
            ax.set_ylabel('Number of Languages')
            ax.legend(title='Signal Type', bbox_to_anchor=(1.05, 1))
            plt.xticks(rotation=30, ha='right')
            plt.tight_layout()
            plt.savefig(os.path.join(EVAL_DIR, '04_signal_by_family.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print('Summary columns:', list(comp_summary.columns))

## 4.3 Does Variant Choice Affect GitHub Alignment?

The key question: does the epistemic stance in the prompt produce translations
that are more or less aligned with community practice?

In [ ]:
if align_df is not None:
    variant_alignment = {}
    for v in VARIANTS:
        col = f'{v}_signal'
        if col in align_df.columns:
            counts = align_df[col].value_counts(normalize=True) * 100
            variant_alignment[v] = counts
    
    if variant_alignment:
        # Build comparison DataFrame
        signal_types = ['strong', 'weak', 'loan', 'absent', 'no_translation']
        rows = []
        for v, counts in variant_alignment.items():
            row = {'variant': v}
            for s in signal_types:
                row[s] = counts.get(s, 0.0)
            rows.append(row)
        comparison_df = pd.DataFrame(rows).set_index('variant')
        
        print('Signal distribution by variant (%):')
        print(comparison_df.round(1).to_string())
        
        # Plot: strong + weak match rate by variant
        fig, ax = plt.subplots(figsize=(9, 5))
        VARIANT_LABELS = {
            'comparative':'Comparative', 'minimal':'Minimal',
            'expert_persona':'Expert Persona', 'contextual':'Contextual',
            'native_rationale':'Native Rationale'
        }
        VARIANT_COLOURS = {
            'comparative':'#4C72B0','minimal':'#DD8452','expert_persona':'#55A868',
            'contextual':'#C44E52','native_rationale':'#8172B2'
        }
        x = range(len(comparison_df))
        ax.bar(x, comparison_df.get('strong', 0), label='Strong match',
               color=[VARIANT_COLOURS.get(v,'grey') for v in comparison_df.index])
        ax.bar(x, comparison_df.get('weak', 0),
               bottom=comparison_df.get('strong', 0), label='Weak match',
               color=[VARIANT_COLOURS.get(v,'grey') for v in comparison_df.index], alpha=0.5)
        ax.set_xticks(list(x))
        ax.set_xticklabels([VARIANT_LABELS.get(v, v) for v in comparison_df.index],
                            rotation=20, ha='right')
        ax.set_ylabel('% of languages')
        ax.set_title('GitHub Alignment Rate by Prompt Variant\n(strong + weak match)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_DIR, '04_alignment_by_variant.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

## 4.4 Deep Dive: Languages with GitHub Data

For languages where we have substantial GitHub data, examine specific cases
where pipeline translations match or diverge from community usage.

In [ ]:
if align_df is not None and 'github_text_count' in align_df.columns:
    # Languages with most GitHub data
    data_rich = align_df.sort_values('github_text_count', ascending=False).head(20)
    print('Languages with most GitHub data:')
    display_cols = ['language_code', 'language_name', 'github_text_count',
                    'comparative_signal', 'comparative_translation']
    available = [c for c in display_cols if c in data_rich.columns]
    print(data_rich[available].to_string(index=False))
    
    # Strong match cases: what did the pipeline correctly predict?
    strong = align_df[align_df.get('comparative_signal', pd.Series()) == 'strong']
    if len(strong) > 0:
        print(f'\nStrong match cases ({len(strong)}):')
        ev_col = 'comparative_evidence'
        for _, row in strong.head(10).iterrows():
            lang = row.get('language_name', row.get('language_code', '?'))
            trans = row.get('comparative_translation', '?')
            evid = row.get(ev_col, '') if ev_col in row else ''
            print(f'  {lang}: "{trans}" → {evid[:80] if evid else "(no evidence snippet)"}')

## 4.5 The GitHub Signal as a DH Presence Map

The distribution of GitHub data across languages is itself evidence about
which communities are present in global DH discourse — separate from whether
their translations are correct. Visualise as a proportional map.

In [ ]:
if align_df is not None and 'github_text_count' in align_df.columns:
    # Group by language family
    LANGUAGE_FAMILIES = {
        'Romance':['es','fr','it','pt','ro','ca'], 'Germanic':['de','nl','sv','da','no','af'],
        'Slavic':['ru','pl','cs','sk','bg','hr','sr','uk'], 'East Asian':['zh','ja','ko'],
        'Semitic':['ar','he'], 'South Asian':['hi','bn','ur','ta','te'],
    }
    def get_family(code):
        for f, codes in LANGUAGE_FAMILIES.items():
            if code in codes: return f
        return 'Other'
    
    align_df['language_family'] = align_df['language_code'].apply(get_family)
    github_by_family = align_df.groupby('language_family')['github_text_count'].agg(
        ['sum', 'mean', 'count']
    ).sort_values('sum', ascending=False)
    
    print('GitHub data presence by language family:')
    print(github_by_family.round(1).to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    github_by_family['sum'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Total GitHub Repo Texts by Language Family')
    axes[0].set_ylabel('Total text count')
    axes[0].tick_params(axis='x', rotation=30)
    
    github_by_family['mean'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
    axes[1].set_title('Mean GitHub Texts per Language')
    axes[1].set_ylabel('Mean text count per language')
    axes[1].tick_params(axis='x', rotation=30)
    
    plt.suptitle('DH Presence on GitHub by Language Family', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, '04_github_presence_by_family.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 4.6 Summary Table for Paper

A clean summary table suitable for inclusion in the DHQ article.

In [ ]:
if align_df is not None:
    paper_rows = []
    for v in VARIANTS:
        col = f'{v}_signal'
        if col not in align_df.columns: continue
        total = len(align_df)
        counts = align_df[col].value_counts()
        paper_rows.append({
            'Variant': v.replace('_', ' ').title(),
            'N Languages': total,
            'Strong Match (%)': round(counts.get('strong', 0) / total * 100, 1),
            'Weak Match (%)':   round(counts.get('weak', 0)   / total * 100, 1),
            'Loan Word (%)':    round(counts.get('loan', 0)   / total * 100, 1),
            'Absent (%)':       round(counts.get('absent', 0) / total * 100, 1),
            'No Translation (%)': round(counts.get('no_translation', 0) / total * 100, 1),
        })
    
    paper_table = pd.DataFrame(paper_rows)
    print('Table for paper:')
    print(paper_table.to_string(index=False))
    
    paper_table.to_csv(os.path.join(EVAL_DIR, '04_paper_summary_table.csv'), index=False)
    print(f'\nSaved to {EVAL_DIR}/04_paper_summary_table.csv')